In [3]:
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
aqi_df = pd.read_csv('../data/daily_aqi_by_county_2025.csv')
temp_df = pd.read_csv('../data/daily_TEMP_2025.csv')
wind_df = pd.read_csv('../data/daily_WIND_2025.csv')


In [5]:
aqi_df_2 = aqi_df.drop(["State Code", "County Code", "Number of Sites Reporting"], axis=1)
aqi_df_3 = aqi_df_2.rename(columns={"county Name" : "County Name",
                                    "Date" : "Date Local",
                                  "Category" : "AQI Category",
                                  "Defining Parameter" : "AQI Defining Parameter",
                                  "Defining Site" : "AQI Defining Site"})

In [6]:
temp_df_2 = temp_df.drop(["State Code", "County Code", "Parameter Code", 
                          "POC", "Datum", "Parameter Name", "Pollutant Standard",
                          "Event Type", "AQI", "City Name", "CBSA Name"], axis=1)
temp_df_3 = temp_df_2.rename(columns={"Sample Duration" : "Temp Sample Duration", 
                                          "Units of Measure" : "Temp Units of Measure",
                                          "Observation Count" : "Temp Observation Count",
                                          "Observation Percent" : "Temp Observation Percent",
                                          "Arithmetic Mean" : "Temp Arithmetic Mean",
                                          "1st Max Value" : "Temp 1st Max Value",
                                          "1st Max Hour" : "Temp 1st Max Hour",
                                          "Method Code" : "Temp Method Code",
                                          "Method Name" : "Temp Method Name",
                                          "Date of Last Change" : "Temp Date of Last Change"})
new_temp_df_col_order = ["State Name", "County Name",
                         "Local Site Name", "Address",
                         "Site Num", "Longitude",
                         "Latitude", "Date Local",
                         "Temp Units of Measure", "Temp Observation Count",
                         "Temp Observation Percent", "Temp Arithmetic Mean",
                         "Temp 1st Max Value", "Temp 1st Max Hour",
                         "Temp Method Code", "Temp Method Name"]

temp_df_clean = temp_df_3[new_temp_df_col_order]

In [7]:
wind_df.info()
wind_df.head(20)

wind_df_2 = wind_df.drop(["State Code", "County Code", "Parameter Code", 
                          "POC", "Datum", "Parameter Name", "Pollutant Standard",
                          "Event Type", "AQI", "City Name", "CBSA Name"], axis=1)
wind_df_3 = wind_df_2.rename(columns={"Sample Duration" : "Temp Sample Duration", 
                                          "Units of Measure" : "Wind Units of Measure",
                                          "Observation Count" : "Wind Observation Count",
                                          "Observation Percent" : "Wind Observation Percent",
                                          "Arithmetic Mean" : "Wind Arithmetic Mean",
                                          "1st Max Value" : "Wind 1st Max Value",
                                          "1st Max Hour" : "Wind 1st Max Hour",
                                          "Method Code" : "Wind Method Code",
                                          "Method Name" : "Wind Method Name",
                                          "Date of Last Change" : "Wind Date of Last Change"})
new_wind_df_col_order = ["State Name", "County Name",
                         "Local Site Name", "Address",
                         "Site Num", "Longitude",
                         "Latitude", "Date Local",
                         "Wind Units of Measure", "Wind Observation Count",
                         "Wind Observation Percent", "Wind Arithmetic Mean",
                         "Wind 1st Max Value", "Wind 1st Max Hour",
                         "Wind Method Code", "Wind Method Name"]

wind_df_clean = wind_df_3[new_wind_df_col_order]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129451 entries, 0 to 129450
Data columns (total 29 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   State Code           129451 non-null  int64  
 1   County Code          129451 non-null  int64  
 2   Site Num             129451 non-null  int64  
 3   Parameter Code       129451 non-null  int64  
 4   POC                  129451 non-null  int64  
 5   Latitude             129451 non-null  float64
 6   Longitude            129451 non-null  float64
 7   Datum                129451 non-null  object 
 8   Parameter Name       129451 non-null  object 
 9   Sample Duration      129451 non-null  object 
 10  Pollutant Standard   0 non-null       float64
 11  Date Local           129451 non-null  object 
 12  Units of Measure     129451 non-null  object 
 13  Event Type           0 non-null       float64
 14  Observation Count    129451 non-null  int64  
 15  Observation Perce

In [8]:
first_merge = pd.merge(temp_df_clean, wind_df_clean, on = ["State Name", "County Name", 
                                             "Local Site Name", "Address", 
                                             "Site Num", "Longitude", 
                                             "Latitude", "Date Local"])

merged_df = pd.merge(first_merge, aqi_df_3, on = ["State Name", "County Name", "Date Local"])

In [9]:
# Defining predictor variables and target variable
X = merged_df[['Wind Observation Count', 'Wind 1st Max Value', 'Wind 1st Max Hour', 'Wind Arithmetic Mean', 
               'Temp Observation Count', 'Temp 1st Max Value', 'Temp 1st Max Hour', 'Temp Arithmetic Mean']].fillna(0)
y = merged_df['AQI']

# Adding a constant for OLS regression
X = sm.add_constant(X)

#Fit for OLS regression model
ols_model = sm.OLS(y, X).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                    AQI   R-squared:                       0.038
Model:                            OLS   Adj. R-squared:                  0.038
Method:                 Least Squares   F-statistic:                     552.6
Date:                Tue, 16 Dec 2025   Prob (F-statistic):               0.00
Time:                        23:22:50   Log-Likelihood:            -5.5197e+05
No. Observations:              111576   AIC:                         1.104e+06
Df Residuals:                  111567   BIC:                         1.104e+06
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     20

In [10]:
X = merged_df[['AQI Category', 'Wind Observation Count', 'Wind 1st Max Value', 'Wind 1st Max Hour', 'Wind Arithmetic Mean', 
               'Temp Observation Count', 'Temp 1st Max Value', 'Temp 1st Max Hour', 'Temp Arithmetic Mean']].fillna(0)
y = merged_df['AQI']
X['AQI Category'] = X['AQI Category'].map({'Unhealthy': 0, 'Moderate': 1, 'Good': 2})

In [11]:
X = X.dropna()
y = y[X.index] 
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Create and train the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

preds = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, preds))
print("R²:", r2_score(y_test, preds))
print("Coefficients:", model.coef_)

MSE: 128.3076617125703
R²: 0.6027264102679755
Coefficients: [-2.70135657e+01  2.83202229e-01  4.63086834e-03 -9.85109587e-03
 -6.32504327e-03 -1.53878742e-01  2.99583777e-01  5.87161569e-02
 -2.23522498e-01]
